# Midori 10.33M physics-informed locomotion policy

This notebook runs the training script from the attached code dataset against the attached MuJoCo episode dataset. It trains a locomotion policy only; voice and camera fusion are later planner stages.

Attach both datasets with **Add Input**, set the paths below to Kaggle's mounted paths, select a GPU accelerator, then Run All. The robot mass is 0.828 kg from the current `mujoco/hello.xml`.

In [ ]:
from pathlib import Path
import subprocess, sys

# These are the Kaggle dataset locations shown in the error you received.
KAGGLE_OWNER_ROOT = Path('/kaggle/input/datasets/amitabhdey101')
EPISODES_ROOT = KAGGLE_OWNER_ROOT / 'episodes'
CODE_DATASET_ROOT = KAGGLE_OWNER_ROOT / 'physics-data'
OUTPUT_ROOT = Path('/kaggle/working/midori-output')
BODY_MASS_KG = 0.828

# Kaggle mounts a dataset as a directory. Find the actual script inside it.
script_matches = list(CODE_DATASET_ROOT.rglob('physics_policy_10m.py'))
if len(script_matches) != 1:
    visible = [str(p.relative_to(CODE_DATASET_ROOT)) for p in CODE_DATASET_ROOT.rglob('*') if p.is_file()][:30] if CODE_DATASET_ROOT.exists() else []
    raise FileNotFoundError(
        f'Expected one physics_policy_10m.py in {CODE_DATASET_ROOT}, found {len(script_matches)}. '
        f'Files found: {visible}. Upload the .py file into the physics-data Kaggle Dataset.'
    )
SCRIPT_PATH = script_matches[0]

assert EPISODES_ROOT.exists(), f'Episode dataset not found: {EPISODES_ROOT}'
print('Episodes:', EPISODES_ROOT)
print('Training script:', SCRIPT_PATH)
print('Output:', OUTPUT_ROOT)


In [ ]:
result = subprocess.run([
    sys.executable, str(SCRIPT_PATH),
    '--episodes', str(EPISODES_ROOT),
    '--output', str(OUTPUT_ROOT),
    '--mass-kg', str(BODY_MASS_KG),
], check=True, text=True)


In [ ]:
import json
summary_path = OUTPUT_ROOT / 'training_summary_10m.json'
print(json.dumps(json.loads(summary_path.read_text()), indent=2))
print('Artifacts:', [p.name for p in OUTPUT_ROOT.iterdir()])

## How to read the result

The saved test error measures imitation of held-out recorded joint targets. It does not prove closed-loop walking. Load the checkpoint with an inference adapter using the saved 64-step context, feature normalization, command order, joint order, and model configuration, then evaluate the policy in MuJoCo with gravity on.